In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/engage-2-value-from-clicks-to-conversions/sample_submission.csv
/kaggle/input/engage-2-value-from-clicks-to-conversions/train_data.csv
/kaggle/input/engage-2-value-from-clicks-to-conversions/test_data.csv


# DATA SET LOADED

In [2]:
train_data = pd.read_csv('/kaggle/input/engage-2-value-from-clicks-to-conversions/train_data.csv')
test_data = pd.read_csv('/kaggle/input/engage-2-value-from-clicks-to-conversions/test_data.csv')

In [3]:
train_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 116023 entries, 0 to 116022
Data columns (total 52 columns):
 #   Column                                        Non-Null Count   Dtype  
---  ------                                        --------------   -----  
 0   trafficSource.isTrueDirect                    42890 non-null   object 
 1   purchaseValue                                 116023 non-null  float64
 2   browser                                       116023 non-null  object 
 3   device.screenResolution                       116023 non-null  object 
 4   trafficSource.adContent                       2963 non-null    object 
 5   trafficSource.keyword                         44162 non-null   object 
 6   screenSize                                    116023 non-null  object 
 7   geoCluster                                    116023 non-null  object 
 8   trafficSource.adwordsClickInfo.slot           4281 non-null    object 
 9   device.mobileDeviceBranding                   11

In [4]:
import warnings
warnings.filterwarnings('ignore')


In [5]:
train_data.head()

,trafficSource.isTrueDirect,purchaseValue,browser,device.screenResolution,trafficSource.adContent,trafficSource.keyword,screenSize,geoCluster,trafficSource.adwordsClickInfo.slot,device.mobileDeviceBranding,...,device.language,deviceType,userChannel,device.browserVersion,totalHits,device.screenColors,sessionStart,geoNetwork.continent,device.isMobile,new_visits
0,NaN,0.0,Edge,not available in demo dataset,NaN,NaN,medium,Region_2,NaN,not available in demo dataset,...,not available in demo dataset,desktop,Social,not available in demo dataset,1,not available in demo dataset,1500100799,Americas,False,1.0
1,True,0.0,Chrome,not available in demo dataset,NaN,NaN,medium,Region_3,NaN,not available in demo dataset,...,not available in demo dataset,desktop,Direct,not available in demo dataset,1,not available in demo dataset,1495262065,Americas,False,1.0
2,True,0.0,Chrome,not available in demo dataset,NaN,(not provided),medium,Region_2,NaN,not available in demo dataset,...,not available in demo dataset,desktop,Organic Search,not available in demo dataset,6,not available in demo dataset,1508510328,Europe,False,NaN
3,NaN,0.0,Internet Explorer,not available in demo dataset,NaN,NaN,medium,Region_4,NaN,not available in demo dataset,...,not available in demo dataset,desktop,Social,not available in demo dataset,1,not available in demo dataset,1483431838,Asia,False,1.0
4,True,88950000.0,Chrome,not available in demo dataset,NaN,NaN,medium,Region_3,NaN,not available in demo dataset,...,not available in demo dataset,desktop,Direct,not available in demo dataset,66,not available in demo dataset,1475804633,Americas,False,1.0


In [6]:
train_data.isnull().sum()

trafficSource.isTrueDirect                       73133
purchaseValue                                        0
browser                                              0
device.screenResolution                              0
trafficSource.adContent                         113060
trafficSource.keyword                            71861
screenSize                                           0
geoCluster                                           0
trafficSource.adwordsClickInfo.slot             111742
device.mobileDeviceBranding                          0
device.mobileInputSelector                           0
userId                                               0
trafficSource.campaign                               0
device.mobileDeviceMarketingName                     0
geoNetwork.networkDomain                             0
gclIdPresent                                         0
device.operatingSystemVersion                        0
sessionNumber                                        0
device.fla

In [7]:

train_data.replace(["not available in demo dataset", "Nan"], np.nan, inplace=True)
test_data.replace(["not available in demo dataset", "Nan"], np.nan, inplace=True)

print("Nulls in train_data:")
print(train_data.isnull().sum())

print("Nulls in test_data:")
print(test_data.isnull().sum())


Nulls in train_data:
trafficSource.isTrueDirect                       73133
purchaseValue                                        0
browser                                              0
device.screenResolution                         116023
trafficSource.adContent                         113060
trafficSource.keyword                            71861
screenSize                                           0
geoCluster                                           0
trafficSource.adwordsClickInfo.slot             111742
device.mobileDeviceBranding                     116023
device.mobileInputSelector                      116023
userId                                               0
trafficSource.campaign                               0
device.mobileDeviceMarketingName                116023
geoNetwork.networkDomain                             0
gclIdPresent                                         0
device.operatingSystemVersion                   116023
sessionNumber                               

In [8]:
missing_percent = train_data.isnull().mean() * 100
cols_to_drop = missing_percent[missing_percent > 90].index
train_data.drop(columns=cols_to_drop, inplace=True)
test_data.drop(columns=cols_to_drop, inplace=True)
print(list(cols_to_drop))


['device.screenResolution', 'trafficSource.adContent', 'trafficSource.adwordsClickInfo.slot', 'device.mobileDeviceBranding', 'device.mobileInputSelector', 'device.mobileDeviceMarketingName', 'device.operatingSystemVersion', 'device.flashVersion', 'geoNetwork.networkLocation', 'trafficSource.adwordsClickInfo.isVideoAd', 'browserMajor', 'device.browserSize', 'trafficSource.adwordsClickInfo.adNetworkType', 'trafficSource.adwordsClickInfo.page', 'device.mobileDeviceModel', 'device.language', 'device.browserVersion', 'device.screenColors']


In [9]:
constant_cols = train_data.columns[train_data.nunique(dropna=False) == 1]
train_data.drop(columns=constant_cols, inplace=True)
test_data.drop(columns=constant_cols, inplace=True)
print(list(constant_cols))


['screenSize', 'totals.visits', 'socialEngagementType', 'locationZone']


In [10]:
if 'trafficSource.isTrueDirect' in train_data.columns:
    train_data['trafficSource.isTrueDirect'].fillna(False, inplace=True)
    train_data['trafficSource.isTrueDirect'] = train_data['trafficSource.isTrueDirect'].astype(bool)

if 'trafficSource.isTrueDirect' in test_data.columns:
    test_data['trafficSource.isTrueDirect'].fillna(False, inplace=True)
    test_data['trafficSource.isTrueDirect'] = test_data['trafficSource.isTrueDirect'].astype(bool)


In [11]:
train_data.replace(["(not set)", "(not provided)"], np.nan, inplace=True)
test_data.replace(["(not set)", "(not provided)"], np.nan, inplace=True)

train_data['trafficSource.medium'].replace("(none)", "direct", inplace=True)
test_data['trafficSource.medium'].replace("(none)", "direct", inplace=True)


In [12]:
if 'new_visits' in train_data.columns:
    train_data['new_visits'].fillna(0, inplace=True)
if 'new_visits' in test_data.columns:
    test_data['new_visits'].fillna(0, inplace=True)

if 'totals.bounces' in train_data.columns:
    train_data['totals.bounces'].fillna(0, inplace=True)
if 'totals.bounces' in test_data.columns:
    test_data['totals.bounces'].fillna(0, inplace=True)


In [13]:
train_data.drop(columns=['trafficSource.referralPath' ,'date','sessionId',
    'trafficSource.keyword',
    'trafficSource.campaign',
    'trafficSource'], errors='ignore', inplace=True)
test_data.drop(columns=['trafficSource.referralPath','date','sessionId',
    'trafficSource.keyword',
    'trafficSource.campaign',
    'trafficSource'], errors='ignore', inplace=True)


In [14]:
print("Train columns:", len(train_data.columns))
print("Test columns:", len(test_data.columns))

print("\nTrain column names:")
print(train_data.columns.tolist())

print("\nTest column names:")
print(test_data.columns.tolist())


Train columns: 24
Test columns: 23

Train column names:
['trafficSource.isTrueDirect', 'purchaseValue', 'browser', 'geoCluster', 'userId', 'geoNetwork.networkDomain', 'gclIdPresent', 'sessionNumber', 'geoNetwork.region', 'os', 'geoNetwork.subContinent', 'trafficSource.medium', 'locationCountry', 'geoNetwork.city', 'geoNetwork.metro', 'pageViews', 'totals.bounces', 'deviceType', 'userChannel', 'totalHits', 'sessionStart', 'geoNetwork.continent', 'device.isMobile', 'new_visits']

Test column names:
['userChannel', 'browser', 'deviceType', 'device.isMobile', 'os', 'geoNetwork.city', 'geoNetwork.continent', 'locationCountry', 'geoNetwork.metro', 'geoNetwork.networkDomain', 'geoNetwork.region', 'geoNetwork.subContinent', 'totals.bounces', 'totalHits', 'new_visits', 'pageViews', 'trafficSource.isTrueDirect', 'trafficSource.medium', 'sessionNumber', 'sessionStart', 'userId', 'geoCluster', 'gclIdPresent']


In [15]:
top_countries = train_data['locationCountry'].value_counts().nlargest(10).index

train_data['locationCountry'] = train_data['locationCountry'].where(train_data['locationCountry'].isin(top_countries), 'Other')
test_data['locationCountry'] = test_data['locationCountry'].where(test_data['locationCountry'].isin(top_countries), 'Other')


In [16]:
train_data['sessionStart'] = pd.to_datetime(train_data['sessionStart'], unit='s', errors='coerce')
test_data['sessionStart'] = pd.to_datetime(test_data['sessionStart'], unit='s', errors='coerce')

train_data['session_hour'] = train_data['sessionStart'].dt.hour
test_data['session_hour'] = test_data['sessionStart'].dt.hour

train_data['session_dayofweek'] = train_data['sessionStart'].dt.dayofweek
test_data['session_dayofweek'] = test_data['sessionStart'].dt.dayofweek


In [17]:
train_data['hits_per_page'] = train_data['totalHits'] / (train_data['pageViews'] + 1)
test_data['hits_per_page'] = test_data['totalHits'] / (test_data['pageViews'] + 1)


In [18]:
user_features = train_data.groupby('userId').agg({
    'sessionNumber': 'count',
    'totalHits': ['mean', 'sum'],
    'pageViews': ['mean', 'sum'],
    'purchaseValue': lambda x: (x > 0).mean(),
    'totals.bounces': 'mean'
}).reset_index()

user_features.columns = ['userId', 'user_session_count', 'user_avg_hits', 'user_total_hits', 
                         'user_avg_pageviews', 'user_total_pageviews', 'user_conversion_rate', 'user_bounce_rate']

train_data = train_data.merge(user_features, on='userId', how='left')
test_data = test_data.merge(user_features.drop(columns='user_conversion_rate'), on='userId', how='left')

for col in ['user_session_count', 'user_avg_hits', 'user_total_hits', 'user_avg_pageviews', 'user_total_pageviews', 'user_bounce_rate']:
    test_data[col].fillna(0, inplace=True)

In [19]:
# user_features = train_data.groupby('userId').agg({
#     'sessionNumber': 'count',
#     'totalHits': 'mean',
#     'pageViews': 'mean',
#     'purchaseValue': lambda x: (x > 0).mean()
# }).reset_index()

# user_features.columns = ['userId', 'user_session_count', 'user_avg_hits', 'user_avg_pageviews', 'user_conversion_rate']

# train_data = train_data.merge(user_features, on='userId', how='left')
# test_data = test_data.merge(user_features.drop(columns='user_conversion_rate'), on='userId', how='left')

# for col in ['user_session_count', 'user_avg_hits', 'user_avg_pageviews']:
#     test_data[col].fillna(0, inplace=True)


In [20]:
train_data['is_weekend'] = train_data['session_dayofweek'].isin([5, 6]).astype(int)
test_data['is_weekend'] = test_data['session_dayofweek'].isin([5, 6]).astype(int)

In [21]:
# train_data['is_engaged'] = ((train_data['pageViews'] > 1) & (train_data['totals.bounces'] == 0)).astype(int)
# test_data['is_engaged'] = ((test_data['pageViews'] > 1) & (test_data['totals.bounces'] == 0)).astype(int)


# train_data['is_new_user'] = ((train_data['new_visits'] > 0) & (train_data['sessionNumber'] == 1)).astype(int)
# test_data['is_new_user'] = ((test_data['new_visits'] > 0) & (test_data['sessionNumber'] == 1)).astype(int)


In [22]:
from xgboost import XGBClassifier, XGBRegressor
from lightgbm import LGBMClassifier, LGBMRegressor
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
import numpy as np
from sklearn.feature_selection import SelectFromModel
from sklearn.model_selection import cross_val_score, RandomizedSearchCV

In [23]:
X = train_data.drop(['purchaseValue', 'user_conversion_rate', 'sessionStart'], axis=1)
y = train_data['purchaseValue']

numeric_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = X.select_dtypes(include=['object', 'bool']).columns.tolist()

# numeric_pipeline = Pipeline([
#     ('imputer', SimpleImputer(strategy='median')),
#     ('scaler', StandardScaler())
# ])

# categorical_pipeline = Pipeline([
#     ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
#     ('encoder', OneHotEncoder(handle_unknown='ignore', sparse=False))
# ])

# preprocessor = ColumnTransformer([
#     ('num', numeric_pipeline, numeric_cols),
#     ('cat', categorical_pipeline, categorical_cols)
# ])
# X_preprocessed = preprocessor.fit_transform(X)
# test_preprocessed = preprocessor.transform(test_data)

y_classifier = (y > 0).astype(int)



In [24]:
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, CatBoostRegressor
from sklearn.model_selection import RandomizedSearchCV, cross_val_score
from sklearn.metrics import f1_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

X = train_data.drop(['purchaseValue', 'user_conversion_rate', 'sessionStart'], axis=1, errors='ignore')
test_X = test_data.drop(['purchaseValue', 'user_conversion_rate', 'sessionStart'], axis=1, errors='ignore')

numeric_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = X.select_dtypes(include=['object', 'bool']).columns.tolist()

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numeric_cols),
    ('cat', 'passthrough', categorical_cols)
])

X_preprocessed = preprocessor.fit_transform(X)
test_preprocessed = preprocessor.transform(test_X)

X_preprocessed_df = pd.DataFrame(X_preprocessed, columns=numeric_cols + categorical_cols)
test_preprocessed_df = pd.DataFrame(test_preprocessed, columns=numeric_cols + categorical_cols)

for col in categorical_cols:
    X_preprocessed_df[col] = X_preprocessed_df[col].astype(str)
    test_preprocessed_df[col] = test_preprocessed_df[col].astype(str)

clf_params = {
    'iterations': [100, 200],
    'depth': [4, 6],
    'learning_rate': [0.05, 0.1],
    'l2_leaf_reg': [3, 5],
    'subsample': [0.8],
    'cat_features': [categorical_cols]
}

reg_params = {
    'iterations': [100, 200],
    'depth': [4, 6],
    'learning_rate': [0.05, 0.1],
    'l2_leaf_reg': [3, 5],
    'subsample': [0.8]
}

clf = CatBoostClassifier(random_state=42, verbose=0, cat_features=categorical_cols)
clf_search = RandomizedSearchCV(
    clf, clf_params, n_iter=8, cv=5, scoring='f1', n_jobs=-1, random_state=42
)
clf_search.fit(X_preprocessed_df, y_classifier)

best_clf = clf_search.best_estimator_
lgb_clf_preds = (best_clf.predict_proba(X_preprocessed_df)[:, 1] > 0.5).astype(int)
test_clf_preds = (best_clf.predict_proba(test_preprocessed_df)[:, 1] > 0.5).astype(int)

y_transformed = np.log1p(y[y > 0])
reg = CatBoostRegressor(random_state=42, verbose=0, cat_features=categorical_cols)
reg_search = RandomizedSearchCV(
    reg, reg_params, n_iter=8, cv=5, scoring='neg_mean_squared_error', n_jobs=-1, random_state=42
)
reg_search.fit(X_preprocessed_df[y > 0], y_transformed)

test_reg_preds = np.expm1(reg_search.best_estimator_.predict(test_preprocessed_df))

max_purchase_value = y[y > 0].quantile(0.95)
test_reg_preds = np.clip(test_reg_preds, 0, max_purchase_value)

final_preds = np.where(test_clf_preds == 1, test_reg_preds, 0)




In [25]:
# pos_weight = (len(y_classifier) - sum(y_classifier)) / sum(y_classifier)  

# clf_params = {
#     'n_estimators': [100, 200, 300, 400],
#     'max_depth': [3, 5, 7, 10],
#     'learning_rate': [0.01, 0.05, 0.1, 0.2],
#     'subsample': [0.6, 0.7, 0.8, 0.9],
#     'colsample_bytree': [0.6, 0.7, 0.8, 0.9],
#     'min_child_weight': [1, 3, 5],
#     'gamma': [0, 0.1, 0.3],
#     'reg_alpha': [0, 0.1, 0.5],
#     'reg_lambda': [0.5, 1, 1.5],
#     'scale_pos_weight': [pos_weight]
# }

# clf = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
# clf_search = RandomizedSearchCV(clf, clf_params, n_iter=20, cv=5, scoring='f1', n_jobs=-1, random_state=42)
# clf_search.fit(X_preprocessed, y_classifier)  

# best_clf = clf_search.best_estimator_

# selector = SelectFromModel(best_clf, prefit=True, threshold='mean')
# X_preprocessed_selected = selector.transform(X_preprocessed)
# test_preprocessed_selected = selector.transform(test_preprocessed)

# best_clf.fit(X_preprocessed_selected, y_classifier)

# reg_params = {
#     'n_estimators': [100, 200, 300, 400],
#     'max_depth': [3, 5, 7, 10],
#     'learning_rate': [0.01, 0.05, 0.1, 0.2],
#     'subsample': [0.6, 0.7, 0.8, 0.9],
#     'colsample_bytree': [0.6, 0.7, 0.8, 0.9],
#     'min_child_weight': [1, 3, 5],
#     'gamma': [0, 0.1, 0.3],
#     'reg_alpha': [0, 0.1, 0.5],
#     'reg_lambda': [0.5, 1, 1.5]
# }

# reg = XGBRegressor(random_state=42)
# reg_search = RandomizedSearchCV(reg, reg_params, n_iter=20, cv=5, scoring='neg_mean_squared_error', n_jobs=-1, random_state=42)
# reg_search.fit(X_preprocessed[y > 0], y[y > 0])

# best_reg = reg_search.best_estimator_

# clf_preds = best_clf.predict(test_preprocessed_selected)
# reg_preds = best_reg.predict(test_preprocessed)

# final_preds = np.where(clf_preds == 1, reg_preds, 0)

In [26]:
submission = pd.DataFrame({
     'id': range(0,test_data.shape[0]),  
     'purchaseValue': final_preds
 })
submission.to_csv('submission.csv', index=False)
print("submission.csv created successfully.")

submission.csv created successfully.
